# Chapter 11 — Neural Networks That Understand Language: King - Man + Woman = ?

**Mata Kuliah:** Deep Learning
**Buku:** Grokking Deep Learning (Andrew W. Trask, Manning)
**Chapter 11:** Neural Networks That Understand Language: King - Man + Woman = ?

Notebook ini berisi reproduksi kode dari buku (diambil dari repository
resmi penulis) beserta penjelasan konsep dan rangkuman chapter.

## Pendahuluan

Chapter 11 membahas **natural language processing (NLP)** dengan
neural network, memakai dataset **IMDB movie reviews** (ulasan film
beserta label positif/negatif).

### Memahami bahasa

NLP memproses teks. Pada **supervised NLP** ulasan diubah menjadi angka,
lalu jaringan memprediksi label. Langkah pertama adalah merepresentasikan
kata sebagai angka dengan menangkap **korelasi kata** dalam input (kata
tertentu berkorelasi dengan sentimen positif atau negatif).

### Embedding layer

Alih-alih memasukkan vektor *one-hot* yang sangat panjang, **embedding
layer** memilih (mengindeks) baris weights untuk kata-kata yang muncul dan
menjumlahkan/merata-ratakannya. Ini ekuivalen dengan perkalian one-hot
tetapi jauh lebih cepat.

### Makna berasal dari loss

Representasi (embedding) kata yang dipelajari bergantung pada **loss**
(tugas belajar). Jika tugasnya memprediksi sentimen, kata-kata yang
mirip sentimen akan berdekatan. Jika tugasnya **mengisi kata yang
hilang** (*fill in the blank*), kata yang mirip secara sintaksis dan
semantik akan berdekatan. Bahkan muncul analogi vektor seperti
**king - man + woman ~ queen**.

## Persiapan Data

File data diunduh dari repository resmi buku agar notebook bisa langsung dijalankan di Colab.

In [ ]:
# Unduh file data dari repository resmi buku (iamtrask/Grokking-Deep-Learning).
# Jika unduhan gagal, unggah file secara manual lewat google.colab.files.upload().
import os, urllib.request

RAW = 'https://raw.githubusercontent.com/iamtrask/Grokking-Deep-Learning/master/'

def ambil(nama):
    folder = os.path.dirname(nama)
    if folder:
        os.makedirs(folder, exist_ok=True)
    if os.path.exists(nama):
        print('sudah ada:', nama)
        return
    try:
        urllib.request.urlretrieve(RAW + nama, nama)
        print('terunduh :', nama)
    except Exception as e:
        print('GAGAL mengunduh', nama, '->', e)
        print('Unggah file tersebut secara manual: from google.colab import files; files.upload()')

for f in ['reviews.txt', 'labels.txt']:
    ambil(f)

# Download the IMDB Dataset

**Memuat dataset IMDB:** `reviews.txt` berisi satu ulasan per baris dan
`labels.txt` berisi label POSITIVE atau NEGATIVE. Ulasan dipecah menjadi
kata, lalu dibuat kosakata (*vocabulary*) dan indeks tiap kata.

In [ ]:
# Download reviews.txt and labels.txt from here: https://github.com/udacity/deep-learning/tree/master/sentiment-network

def pretty_print_review_and_label(i):
   print(labels[i] + "\t:\t" + reviews[i][:80] + "...")

g = open('reviews.txt','r') # What we know!
reviews = list(map(lambda x:x[:-1],g.readlines()))
g.close()

g = open('labels.txt','r') # What we WANT to know!
labels = list(map(lambda x:x[:-1].upper(),g.readlines()))
g.close()

# Capturing Word Correlation in Input Data

**Menangkap korelasi kata:** tiap ulasan dinyatakan sebagai daftar indeks
kata yang muncul. Jaringan menjumlahkan weights kata-kata tersebut
sehingga kata dengan weight positif mendorong sentimen positif dan
sebaliknya.

In [ ]:
import numpy as np

onehots = {}
onehots['cat'] = np.array([1,0,0,0])
onehots['the'] = np.array([0,1,0,0])
onehots['dog'] = np.array([0,0,1,0])
onehots['sat'] = np.array([0,0,0,1])

sentence = ['the','cat','sat']
x = onehots[sentence[0]] + \
    onehots[sentence[1]] + \
    onehots[sentence[2]]

print("Sent Encoding:" + str(x))

# Predicting Movie Reviews

**Memprediksi ulasan:** jaringan dua layer (embedding + output sigmoid)
dilatih dengan gradient descent. Akurasi uji mencapai sekitar 85% pada
percobaan buku. Karena hanya mengindeks baris weights, training sangat
cepat.

In [ ]:
import sys

f = open('reviews.txt')
raw_reviews = f.readlines()
f.close()

f = open('labels.txt')
raw_labels = f.readlines()
f.close()

tokens = list(map(lambda x:set(x.split(" ")),raw_reviews))

vocab = set()
for sent in tokens:
    for word in sent:
        if(len(word)>0):
            vocab.add(word)
vocab = list(vocab)

word2index = {}
for i,word in enumerate(vocab):
    word2index[word]=i

input_dataset = list()
for sent in tokens:
    sent_indices = list()
    for word in sent:
        try:
            sent_indices.append(word2index[word])
        except:
            ""
    input_dataset.append(list(set(sent_indices)))

target_dataset = list()
for label in raw_labels:
    if label == 'positive\n':
        target_dataset.append(1)
    else:
        target_dataset.append(0)

In [ ]:
import numpy as np
np.random.seed(1)

def sigmoid(x):
    return 1/(1 + np.exp(-x))

alpha, iterations = (0.01, 2)
hidden_size = 100

weights_0_1 = 0.2*np.random.random((len(vocab),hidden_size)) - 0.1
weights_1_2 = 0.2*np.random.random((hidden_size,1)) - 0.1

correct,total = (0,0)
for iter in range(iterations):
    
    # train on first 24,000
    for i in range(len(input_dataset)-1000):

        x,y = (input_dataset[i],target_dataset[i])
        layer_1 = sigmoid(np.sum(weights_0_1[x],axis=0)) #embed + sigmoid
        layer_2 = sigmoid(np.dot(layer_1,weights_1_2)) # linear + softmax

        layer_2_delta = layer_2 - y # compare pred with truth
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) #backprop

        weights_0_1[x] -= layer_1_delta * alpha
        weights_1_2 -= np.outer(layer_1,layer_2_delta) * alpha

        if(np.abs(layer_2_delta) < 0.5):
            correct += 1
        total += 1
        if(i % 10 == 9):
            progress = str(i/float(len(input_dataset)))
            sys.stdout.write('\rIter:'+str(iter)\
                             +' Progress:'+progress[2:4]\
                             +'.'+progress[4:6]\
                             +'% Training Accuracy:'\
                             + str(correct/float(total)) + '%')
    print()
correct,total = (0,0)
for i in range(len(input_dataset)-1000,len(input_dataset)):

    x = input_dataset[i]
    y = target_dataset[i]

    layer_1 = sigmoid(np.sum(weights_0_1[x],axis=0))
    layer_2 = sigmoid(np.dot(layer_1,weights_1_2))
    
    if(np.abs(layer_2 - y) < 0.5):
        correct += 1
    total += 1
print("Test Accuracy:" + str(correct / float(total)))

In [ ]:
tokens[0]

# Comparing Word Embeddings

**Membandingkan embedding kata:** setelah training, kata yang embedding-nya
paling dekat (berdasarkan jarak Euclidean negatif) dengan kata tertentu
ditampilkan, misalnya kata dengan makna sentimen serupa.

In [ ]:
from collections import Counter
import math 

def similar(target='beautiful'):
    target_index = word2index[target]
    scores = Counter()
    for word,index in word2index.items():
        raw_difference = weights_0_1[index] - (weights_0_1[target_index])
        squared_difference = raw_difference * raw_difference
        scores[word] = -math.sqrt(sum(squared_difference))

    return scores.most_common(10)

In [ ]:
print(similar('beautiful'))

In [ ]:
print(similar('terrible'))

# Filling in the Blank

**Mengisi kata yang hilang:** loss diubah menjadi memprediksi kata yang
dihilangkan dari konteks sekitarnya. Embedding yang dihasilkan menangkap
kemiripan makna antar kata (kata sejenis berdekatan), berbeda dengan
embedding pada tugas sentimen.

In [ ]:
import sys,random,math
from collections import Counter
import numpy as np

np.random.seed(1)
random.seed(1)
f = open('reviews.txt')
raw_reviews = f.readlines()
f.close()

tokens = list(map(lambda x:(x.split(" ")),raw_reviews))
wordcnt = Counter()
for sent in tokens:
    for word in sent:
        wordcnt[word] -= 1
vocab = list(set(map(lambda x:x[0],wordcnt.most_common())))

word2index = {}
for i,word in enumerate(vocab):
    word2index[word]=i

concatenated = list()
input_dataset = list()
for sent in tokens:
    sent_indices = list()
    for word in sent:
        try:
            sent_indices.append(word2index[word])
            concatenated.append(word2index[word])
        except:
            ""
    input_dataset.append(sent_indices)
concatenated = np.array(concatenated)
random.shuffle(input_dataset)

In [ ]:
alpha, iterations = (0.05, 2)
hidden_size,window,negative = (50,2,5)

weights_0_1 = (np.random.rand(len(vocab),hidden_size) - 0.5) * 0.2
weights_1_2 = np.random.rand(len(vocab),hidden_size)*0

layer_2_target = np.zeros(negative+1)
layer_2_target[0] = 1

def similar(target='beautiful'):
  target_index = word2index[target]

  scores = Counter()
  for word,index in word2index.items():
    raw_difference = weights_0_1[index] - (weights_0_1[target_index])
    squared_difference = raw_difference * raw_difference
    scores[word] = -math.sqrt(sum(squared_difference))
  return scores.most_common(10)

def sigmoid(x):
    return 1/(1 + np.exp(-x))

for rev_i,review in enumerate(input_dataset * iterations):
  for target_i in range(len(review)):
        
    # since it's really expensive to predict every vocabulary
    # we're only going to predict a random subset
    target_samples = [review[target_i]]+list(concatenated\
    [(np.random.rand(negative)*len(concatenated)).astype('int').tolist()])

    left_context = review[max(0,target_i-window):target_i]
    right_context = review[target_i+1:min(len(review),target_i+window)]

    layer_1 = np.mean(weights_0_1[left_context+right_context],axis=0)
    layer_2 = sigmoid(layer_1.dot(weights_1_2[target_samples].T))
    layer_2_delta = layer_2 - layer_2_target
    layer_1_delta = layer_2_delta.dot(weights_1_2[target_samples])

    weights_0_1[left_context+right_context] -= layer_1_delta * alpha
    weights_1_2[target_samples] -= np.outer(layer_2_delta,layer_1)*alpha

  if(rev_i % 250 == 0):
    sys.stdout.write('\rProgress:'+str(rev_i/float(len(input_dataset)
        *iterations)) + "   " + str(similar('terrible')))
  sys.stdout.write('\rProgress:'+str(rev_i/float(len(input_dataset)
        *iterations)))
print(similar('terrible'))

# King - Man + Woman ~= Queen

**Analogi kata:** embedding yang dilatih pada tugas mengisi kata dapat
melakukan aritmetika vektor, misalnya `king - man + woman ~= queen`.
Analogi ini menunjukkan bahwa hubungan makna (gender, tempat, dan lain-lain)
terkodekan sebagai arah di ruang embedding.

In [ ]:
def analogy(positive=['terrible','good'],negative=['bad']):
    
    norms = np.sum(weights_0_1 * weights_0_1,axis=1)
    norms.resize(norms.shape[0],1)
    
    normed_weights = weights_0_1 * norms
    
    query_vect = np.zeros(len(weights_0_1[0]))
    for word in positive:
        query_vect += normed_weights[word2index[word]]
    for word in negative:
        query_vect -= normed_weights[word2index[word]]
    
    scores = Counter()
    for word,index in word2index.items():
        raw_difference = weights_0_1[index] - query_vect
        squared_difference = raw_difference * raw_difference
        scores[word] = -math.sqrt(sum(squared_difference))
        
    return scores.most_common(10)[1:]

In [ ]:
analogy(['terrible','good'],['bad'])

In [ ]:
analogy(['elizabeth','he'],['she'])

## Key Ideas

- Kata diubah menjadi indeks dan embedding berupa baris weights.
- Embedding layer = mengindeks weights, setara dengan one-hot dot weights tetapi lebih cepat.
- Loss function menentukan makna yang ditangkap embedding.
- Embedding dapat membentuk analogi vektor (king - man + woman ~ queen).

## Kesimpulan


Chapter 11 menjelaskan bagaimana neural network mempelajari bahasa pada
tingkat kata menggunakan **word embeddings**, dan bahwa pilihan **loss
function** menentukan sifat yang ditangkap embedding (sentimen vs makna
umum). Puncaknya adalah fenomena analogi kata, misalnya *king - man +
woman ~ queen*. Pemahaman tentang pembuatan dan penyetelan loss akan
sangat penting pada chapter-chapter berikutnya.